# Supervisor and Hierarchical Workflows

A **supervisor** is an orchestrator that **delegates** work to specialist agents instead of doing every task itself. In a **hierarchical workflow**, supervisors form a tree: a top-level triage agent routes to department supervisors, who route to leaf workers with narrow tools.

This pattern appears in customer support hubs, engineering orgs, and research pipelines — anywhere expertise is split by domain and you need clear accountability at each level.

This notebook covers:

- **Flat supervisor** vs **hierarchical supervisor** topologies
- **Handoff contracts** between levels
- A working **ShopCo Support Hub** with three departments and leaf workers
- **Escalation** when a worker cannot resolve a request
- **Tracing** across the full hierarchy

Everything is self-contained plain Python — no frameworks required.

In [ ]:
"""
ShopCo Customer Support Hub — hierarchical supervisor environment.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {"customer": "alice@shop.com", "status": "shipped", "amount": 1299.00},
    "ORD-1002": {"customer": "bob@shop.com", "status": "processing", "amount": 449.00},
}

POLICIES = [
    {"id": "ret-02", "text": "Returns within 30 days if unused."},
    {"id": "cancel-03", "text": "Cancellations allowed only before shipment."},
    {"id": "bill-04", "text": "Billing disputes resolved within 5 business days."},
]

BILLING_ACCOUNTS = {
    "alice@shop.com": {"plan": "pro", "balance": 0.00, "payment_method": "visa-4242"},
    "bob@shop.com": {"plan": "basic", "balance": 12.50, "payment_method": "master-8888"},
}

ESCALATIONS: list[dict[str, Any]] = []

print(f"Orders: {len(ORDERS)} | Policies: {len(POLICIES)} | Billing accounts: {len(BILLING_ACCOUNTS)}")

---

## 1. The Supervisor Pattern

In a supervisor workflow, one agent (or routing node) decides **who acts next** — it does not execute every tool itself.

```
User query
    │
    ▼
Supervisor ──► "route to orders_worker"
    │
    ▼
Worker executes tools ──► observation ──► back to Supervisor
    │
    ▼
Supervisor ──► "done" or "route to policy_worker"
```

| Role | Responsibility |
|------|----------------|
| **Supervisor** | Classify intent, pick worker, merge results, decide termination |
| **Worker** | Narrow tools + focused prompt; executes delegated subtask |
| **Orchestrator** | Code that runs the graph — enforces max steps, logging, handoffs |

The supervisor can be an LLM with structured output (`{"next": "orders_worker"}`) or a rule-based router for deterministic triage.

---

## 2. Flat vs Hierarchical Supervisors

**Flat supervisor** — one supervisor routes directly to all workers:

```
Top Supervisor ──┬── order_worker
               ├── policy_worker
               └── billing_worker
```

**Hierarchical supervisor** — supervisors delegate to other supervisors:

```
                    Top Supervisor (triage)
                           │
        ┌──────────────────┼──────────────────┐
        ▼                  ▼                  ▼
 Orders Supervisor   Policy Supervisor   Billing Supervisor
        │                  │                  │
        ▼                  ▼                  ▼
   order_worker       policy_worker      billing_worker
```

| | Flat | Hierarchical |
|--|------|--------------|
| Routing complexity | All workers visible to top supervisor | Each level sees only its subtree |
| Tool surface per node | Large menu at top | Small menu at each level |
| Org alignment | Small teams | Mirrors departments / domains |
| Latency | Fewer hops | Extra routing steps |
| Best for | ≤5 workers, simple domains | Many specialists, clear org boundaries |

---

## 3. Hierarchy Levels in ShopCo

We model a three-level support hub:

| Level | Agent | Routes to |
|-------|-------|-----------|
| L0 | `top_supervisor` | Department supervisors |
| L1 | `orders_supervisor`, `policy_supervisor`, `billing_supervisor` | Leaf workers |
| L2 | `order_worker`, `policy_worker`, `billing_worker` | Tools only |

Each worker has **one or two tools** — least privilege at the leaf level.

In [ ]:
class Department(str, Enum):
    ORDERS = "orders"
    POLICY = "policy"
    BILLING = "billing"
    UNKNOWN = "unknown"


HIERARCHY: dict[str, list[str]] = {
    "top_supervisor": ["orders_supervisor", "policy_supervisor", "billing_supervisor"],
    "orders_supervisor": ["order_worker"],
    "policy_supervisor": ["policy_worker"],
    "billing_supervisor": ["billing_worker"],
}

WORKER_TOOLS: dict[str, list[str]] = {
    "order_worker": ["lookup_order"],
    "policy_worker": ["search_policy"],
    "billing_worker": ["lookup_billing", "search_policy"],
}

SUPERVISOR_OF: dict[str, str] = {
    "orders_supervisor": "top_supervisor",
    "policy_supervisor": "top_supervisor",
    "billing_supervisor": "top_supervisor",
    "order_worker": "orders_supervisor",
    "policy_worker": "policy_supervisor",
    "billing_worker": "billing_supervisor",
}

print("Hierarchy:")
for sup, children in HIERARCHY.items():
    print(f"  {sup} → {children}")

---

## 4. Handoff Contracts Between Levels

A handoff is not a bare function call. Each level passes a **contract** downstream:

- `from_agent` / `to_agent` — who delegated and who receives
- `task` — distilled subtask (not the full chat history)
- `constraints` — read-only, cite sources, no billing changes
- `context` — structured fields extracted upstream (order_id, email)

Workers return a **WorkerResult**; supervisors decide whether to escalate, re-route, or finish.

In [ ]:
@dataclass
class Handoff:
    handoff_id: str
    from_agent: str
    to_agent: str
    task: str
    constraints: list[str] = field(default_factory=list)
    context: dict[str, Any] = field(default_factory=dict)
    depth: int = 0


@dataclass
class WorkerResult:
    success: bool
    answer: str
    observation: str = ""
    escalate: bool = False
    escalate_reason: str = ""


@dataclass
class HierarchyRunResult:
    answer: str
    handoffs: list[Handoff]
    trace: list[str]
    departments_visited: list[str]
    escalated: bool = False
    steps: int = 0


def make_handoff(from_agent: str, to_agent: str, task: str, depth: int, **ctx: Any) -> Handoff:
    return Handoff(
        handoff_id=str(uuid.uuid4())[:8],
        from_agent=from_agent,
        to_agent=to_agent,
        task=task,
        constraints=["read_only", "cite_policy_ids"],
        context=dict(ctx),
        depth=depth,
    )


example = make_handoff(
    "top_supervisor", "orders_supervisor",
    "Customer asking about order ORD-1001 status", depth=1, order_id="ORD-1001"
)
print(pretty(example))

---

## 5. L2 Tools — Leaf Integrations

Workers call tools. Observations use `[STATUS: ok]` / `[STATUS: error]` so supervisors can parse failures.

In [ ]:
def lookup_order(order_id: str) -> str:
    oid = order_id.upper().strip()
    if not re.match(r"^ORD-\d{4}$", oid):
        return f"[STATUS: error] Invalid order_id '{order_id}'"
    order = ORDERS.get(oid)
    if not order:
        return f"[STATUS: error] Order {oid} not found"
    return f"[STATUS: ok] {json.dumps({'order_id': oid, **order})}"


def search_policy(query: str) -> str:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    hits = [p for p in POLICIES if any(t in p["text"].lower() for t in terms)]
    return f"[STATUS: ok] {json.dumps(hits or POLICIES[:1])}"


def lookup_billing(email: str) -> str:
    acct = BILLING_ACCOUNTS.get(email.lower())
    if not acct:
        return f"[STATUS: error] No billing account for {email}"
    return f"[STATUS: ok] {json.dumps({'email': email, **acct})}"


TOOL_IMPL: dict[str, Callable[..., str]] = {
    "lookup_order": lookup_order,
    "search_policy": search_policy,
    "lookup_billing": lookup_billing,
}

print(lookup_order("ORD-1001"))
print(lookup_billing("alice@shop.com"))

---

## 6. L2 Workers — Leaf Agents

Each worker receives a handoff, calls its tools, and returns a `WorkerResult`. Workers **do not** route to other agents.

In [ ]:
def extract_order_id(text: str) -> str | None:
    match = re.search(r"ORD-\d{4}", text.upper())
    return match.group(0) if match else None


def extract_email(text: str) -> str | None:
    match = re.search(r"[\w.-]+@[\w.-]+\.[a-zA-Z]{2,}", text)
    return match.group(0).lower() if match else None


def run_order_worker(handoff: Handoff) -> WorkerResult:
    order_id = handoff.context.get("order_id") or extract_order_id(handoff.task) or "ORD-1001"
    obs = TOOL_IMPL["lookup_order"](order_id=order_id)
    if "[STATUS: error]" in obs:
        return WorkerResult(False, obs, obs, escalate=True, escalate_reason="order_not_found")
    data = json.loads(obs.replace("[STATUS: ok] ", ""))
    answer = (
        f"Order {data['order_id']} is {data['status']} "
        f"(${data['amount']:.2f}) for {data['customer']}."
    )
    return WorkerResult(True, answer, obs)


def run_policy_worker(handoff: Handoff) -> WorkerResult:
    obs = TOOL_IMPL["search_policy"](query=handoff.task)
    policies = json.loads(obs.replace("[STATUS: ok] ", ""))
    cited = " | ".join(f"[{p['id']}] {p['text']}" for p in policies)
    return WorkerResult(True, f"Per ShopCo policy: {cited}", obs)


def run_billing_worker(handoff: Handoff) -> WorkerResult:
    email = handoff.context.get("email") or extract_email(handoff.task)
    if not email:
        return WorkerResult(
            False, "Need customer email for billing lookup.",
            escalate=True, escalate_reason="missing_email",
        )
    obs = TOOL_IMPL["lookup_billing"](email=email)
    if "[STATUS: error]" in obs:
        return WorkerResult(False, obs, obs, escalate=True, escalate_reason="billing_not_found")
    data = json.loads(obs.replace("[STATUS: ok] ", ""))
    policy_obs = TOOL_IMPL["search_policy"](query="billing dispute")
    policies = json.loads(policy_obs.replace("[STATUS: ok] ", ""))
    policy_cite = policies[0]["id"] if policies else "bill-04"
    answer = (
        f"Account {data['email']}: plan={data['plan']}, balance=${data['balance']:.2f}. "
        f"Disputes per [{policy_cite}]."
    )
    return WorkerResult(True, answer, obs)


WORKERS: dict[str, Callable[[Handoff], WorkerResult]] = {
    "order_worker": run_order_worker,
    "policy_worker": run_policy_worker,
    "billing_worker": run_billing_worker,
}

demo_handoff = make_handoff("orders_supervisor", "order_worker", "ORD-1001 status?", 2, order_id="ORD-1001")
print(run_order_worker(demo_handoff).answer)

---

## 7. L1 Department Supervisors

Department supervisors receive a handoff from the top supervisor, pick the right worker, and return the worker result upward. If the worker escalates, the department supervisor can request human help.

In [ ]:
DEPARTMENT_WORKER: dict[str, str] = {
    "orders_supervisor": "order_worker",
    "policy_supervisor": "policy_worker",
    "billing_supervisor": "billing_worker",
}


def run_department_supervisor(
    supervisor_id: str,
    handoff: Handoff,
    trace: list[str],
    handoffs: list[Handoff],
) -> WorkerResult:
    worker_id = DEPARTMENT_WORKER[supervisor_id]
    worker_handoff = make_handoff(
        supervisor_id, worker_id, handoff.task, handoff.depth + 1, **handoff.context
    )
    handoffs.append(worker_handoff)
    trace.append(f"{supervisor_id} → {worker_id}")

    result = WORKERS[worker_id](worker_handoff)
    trace.append(f"{worker_id}: success={result.success}")

    if result.escalate:
        ESCALATIONS.append({
            "from": supervisor_id,
            "reason": result.escalate_reason,
            "task": handoff.task,
            "ts": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        })
        trace.append(f"ESCALATE from {supervisor_id}: {result.escalate_reason}")
    return result

---

## 8. L0 Top Supervisor — Triage Router

The top supervisor classifies the customer query into a **department** and delegates to the matching department supervisor. This is the entry point for every support request.

In [ ]:
def classify_department(query: str) -> Department:
    """Rule-based triage — production would use LLM with structured output."""
    q = query.lower()
    if any(w in q for w in ("bill", "charge", "invoice", "payment", "refund", "balance")):
        return Department.BILLING
    if any(w in q for w in ("return", "cancel", "policy", "shipping", "warranty")):
        return Department.POLICY
    if "ord-" in q or "order" in q or "track" in q or "delivery" in q:
        return Department.ORDERS
    return Department.POLICY  # default general FAQ


DEPARTMENT_SUPERVISOR: dict[Department, str] = {
    Department.ORDERS: "orders_supervisor",
    Department.POLICY: "policy_supervisor",
    Department.BILLING: "billing_supervisor",
}


def extract_context(query: str) -> dict[str, Any]:
    ctx: dict[str, Any] = {}
    oid = extract_order_id(query)
    if oid:
        ctx["order_id"] = oid
    email = extract_email(query)
    if email:
        ctx["email"] = email
    return ctx


for q in [
    "Where is order ORD-1001?",
    "What is your return policy?",
    "I was charged twice on my invoice for alice@shop.com",
]:
    dept = classify_department(q)
    print(f"{dept.value:<8} → {DEPARTMENT_SUPERVISOR[dept]}  |  {q[:50]}")

---

## 9. Hierarchical Orchestrator — Full Run

`HierarchicalSupportOrchestrator` wires all three levels: top supervisor → department supervisor → worker → answer.

In [ ]:
MAX_HIERARCHY_STEPS = 10


class HierarchicalSupportOrchestrator:
    def __init__(self, max_steps: int = MAX_HIERARCHY_STEPS):
        self.max_steps = max_steps

    def run(self, user_query: str, thread_id: str = "t-001") -> HierarchyRunResult:
        trace: list[str] = [f"session:{thread_id}"]
        handoffs: list[Handoff] = []
        steps = 0

        # L0: Top supervisor triage
        steps += 1
        dept = classify_department(user_query)
        dept_supervisor = DEPARTMENT_SUPERVISOR[dept]
        ctx = extract_context(user_query)
        trace.append(f"top_supervisor: dept={dept.value} → {dept_supervisor}")

        top_handoff = make_handoff(
            "top_supervisor", dept_supervisor, user_query, depth=1, **ctx
        )
        handoffs.append(top_handoff)

        # L1: Department supervisor delegates to worker
        steps += 1
        result = run_department_supervisor(dept_supervisor, top_handoff, trace, handoffs)

        if steps >= self.max_steps:
            return HierarchyRunResult(
                "Max steps exceeded", handoffs, trace, [dept.value], steps=steps
            )

        # L0: Top supervisor merges (pass-through for single-worker path)
        if result.escalate:
            answer = (
                f"I could not fully resolve your request ({result.escalate_reason}). "
                f"A human agent will follow up. Reference: {handoffs[-1].handoff_id}"
            )
            trace.append("top_supervisor: escalated to human")
            return HierarchyRunResult(
                answer, handoffs, trace, [dept.value], escalated=True, steps=steps
            )

        trace.append("top_supervisor: final_answer")
        return HierarchyRunResult(
            result.answer, handoffs, trace, [dept.value], steps=steps
        )


orchestrator = HierarchicalSupportOrchestrator()

TEST_QUERIES = [
    "Where is my order ORD-1001?",
    "Can I return an item after 30 days?",
    "Why was I charged $12.50? My email is bob@shop.com",
    "Lookup order ORD-9999 please",  # triggers escalation
]

for q in TEST_QUERIES:
    ESCALATIONS.clear()
    r = orchestrator.run(q)
    print(f"\nQ: {q}")
    print(f"A: {r.answer}")
    print(f"   dept={r.departments_visited} | steps={r.steps} | handoffs={len(r.handoffs)} | escalated={r.escalated}")

---

## 10. Trace the Full Hierarchy

Production systems log every hop. Inspect handoffs and trace events for debugging and evals.

In [ ]:
result = orchestrator.run("Where is order ORD-1002 and can I still cancel?")

print("Handoff chain:")
for h in result.handoffs:
    indent = "  " * h.depth
    print(f"{indent}{h.from_agent} → {h.to_agent}  [{h.handoff_id}]  ctx={h.context}")

print("\nTrace:")
for line in result.trace:
    print(f"  {line}")

print(f"\nAnswer: {result.answer}")

---

## 11. Flat Supervisor for Comparison

The same ShopCo hub implemented as a **flat** supervisor — one router, three workers, no department layer.

In [ ]:
FLAT_ROUTING: dict[Department, str] = {
    Department.ORDERS: "order_worker",
    Department.POLICY: "policy_worker",
    Department.BILLING: "billing_worker",
}


class FlatSupportOrchestrator:
    def run(self, user_query: str) -> HierarchyRunResult:
        trace = ["flat_supervisor: triage"]
        handoffs: list[Handoff] = []
        dept = classify_department(user_query)
        worker_id = FLAT_ROUTING[dept]
        ctx = extract_context(user_query)

        h = make_handoff("flat_supervisor", worker_id, user_query, depth=1, **ctx)
        handoffs.append(h)
        trace.append(f"flat_supervisor → {worker_id}")

        result = WORKERS[worker_id](h)
        trace.append(f"{worker_id}: success={result.success}")
        return HierarchyRunResult(
            result.answer if result.success else result.answer,
            handoffs,
            trace,
            [dept.value],
            escalated=result.escalate,
            steps=2,
        )


flat = FlatSupportOrchestrator()
hier = HierarchicalSupportOrchestrator()

q = "What is order ORD-1001 status?"
r_flat = flat.run(q)
r_hier = hier.run(q)

print(f"Flat:        steps={r_flat.steps}  handoffs={len(r_flat.handoffs)}  trace={r_flat.trace}")
print(f"Hierarchical: steps={r_hier.steps}  handoffs={len(r_hier.handoffs)}  trace={r_hier.trace}")

---

## 12. Multi-Department Queries — Supervisor Loops

Some queries span departments ("order status AND return policy"). The top supervisor can **loop**: delegate to multiple department supervisors and merge results.

In [ ]:
def departments_for_query(query: str) -> list[Department]:
    """Detect multi-intent queries needing multiple departments."""
    q = query.lower()
    depts: list[Department] = []
    if "ord-" in q or "order" in q:
        depts.append(Department.ORDERS)
    if any(w in q for w in ("return", "cancel", "policy")):
        depts.append(Department.POLICY)
    if any(w in q for w in ("bill", "charge", "invoice")):
        depts.append(Department.BILLING)
    return depts or [Department.POLICY]


class MultiDepartmentOrchestrator:
    """Top supervisor loops over departments and merges worker answers."""

    def run(self, user_query: str) -> HierarchyRunResult:
        trace = ["top_supervisor: multi-dept mode"]
        handoffs: list[Handoff] = []
        answers: list[str] = []
        depts_visited: list[str] = []
        steps = 0

        for dept in departments_for_query(user_query):
            steps += 1
            sup = DEPARTMENT_SUPERVISOR[dept]
            depts_visited.append(dept.value)
            ctx = extract_context(user_query)
            h = make_handoff("top_supervisor", sup, user_query, depth=1, **ctx)
            handoffs.append(h)
            trace.append(f"top_supervisor → {sup} (dept={dept.value})")
            result = run_department_supervisor(sup, h, trace, handoffs)
            if result.success:
                answers.append(result.answer)

        merged = " ".join(answers) if answers else "Unable to resolve query."
        trace.append("top_supervisor: merged answers")
        return HierarchyRunResult(merged, handoffs, trace, depts_visited, steps=steps)


multi = MultiDepartmentOrchestrator()
r = multi.run("Where is order ORD-1002 and can I still cancel it?")
print(f"Departments: {r.departments_visited}")
print(f"Handoffs: {len(r.handoffs)} | Steps: {r.steps}")
print(f"Answer: {r.answer}")

---

## 13. Termination and Escalation Rules

| Signal | Who decides | Action |
|--------|-------------|--------|
| Worker success | Department supervisor | Return answer up |
| Worker escalate | Department supervisor | Log + bubble to top |
| Top supervisor merge | Top supervisor | Combine multi-dept answers |
| Max steps | Orchestrator | Stop with partial answer |
| Human required | Top supervisor | Create escalation ticket |

Escalation should **never silently fail** — always log reason and handoff id for human agents.

In [ ]:
ESCALATIONS.clear()
_ = orchestrator.run("Please lookup order ORD-9999")

print("Escalation queue:")
for e in ESCALATIONS:
    print(f"  [{e['ts']}] {e['from']}: {e['reason']} — {e['task'][:50]}")

---

## 14. Design Guidelines

| Guideline | Rationale |
|-----------|-----------|
| **One supervisor, one routing decision** per hop | Avoid multi-tool supervisors |
| **Workers are leaf nodes** | Workers call tools; they do not route |
| **Shrink tool menus per level** | Top sees departments; workers see 1–2 tools |
| **Explicit handoff contracts** | Pass distilled task + context, not full chat log |
| **Log every hop** | Hierarchy bugs hide in delegation chains |
| **Cap total steps** | Prevent infinite supervisor loops |
| **Start flat, add hierarchy when needed** | Extra hops cost latency |

---

## 15. Anti-Patterns

| Anti-pattern | Problem | Fix |
|--------------|---------|-----|
| **Supervisor with all tools** | Defeats purpose of delegation | Move tools to workers |
| **Workers routing to workers** | Hidden graph, no audit trail | Only supervisors route |
| **Passing full chat history** | Context bloat, injection risk | Distill handoff task + structured context |
| **Too many hierarchy levels** | Latency, routing errors | Flatten or merge departments |
| **No escalation path** | User stuck on tool failure | Bubble escalate flag to human queue |

---

## 16. Optional — LLM Supervisor Router

Replace `classify_department()` with an LLM that returns structured JSON when `USE_LIVE_LLM = True`.

In [ ]:
def classify_department_llm(query: str) -> Department:
    from openai import OpenAI

    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Classify customer support queries into exactly one department: "
                    "orders, policy, or billing. Reply with only the department name."
                ),
            },
            {"role": "user", "content": query},
        ],
        temperature=0,
    )
    label = resp.choices[0].message.content.strip().lower()
    mapping = {
        "orders": Department.ORDERS,
        "policy": Department.POLICY,
        "billing": Department.BILLING,
    }
    return mapping.get(label, Department.POLICY)


if USE_LIVE_LLM:
    for q in ["I need a refund on my last charge", "Track order ORD-1001"]:
        print(classify_department_llm(q), "←", q)
else:
    print("Offline mode — rule-based classify_department() used in demos.")

---

## 17. Check Your Understanding

1. What is the difference between an **orchestrator** and a **supervisor**?
2. Why use a **hierarchical** supervisor instead of a flat one?
3. Why should **workers** not route to other agents?
4. What belongs in a **handoff contract** between supervisor levels?
5. How does **escalation** propagate from worker to human?
6. When should a top supervisor **loop** across multiple departments?
7. Name two supervisor anti-patterns and their fixes.

---

## 18. Summary

Supervisor workflows delegate cognitive and tool work to specialists. **Hierarchical** supervisors mirror org structure: top triage → department supervisors → leaf workers.

| Concept | ShopCo implementation |
|---------|----------------------|
| Top supervisor | `classify_department()` → department supervisor |
| Department supervisor | Delegates to one worker |
| Worker | `order_worker`, `policy_worker`, `billing_worker` |
| Handoff | `Handoff` dataclass with task + context |
| Escalation | `WorkerResult.escalate` → human queue |
| Multi-dept | `MultiDepartmentOrchestrator` loops and merges |

Start with a **flat** supervisor when you have few workers. Add hierarchy when routing menus grow, teams own domains, or compliance requires departmental boundaries. Always log every hop — in a tree, bugs hide in the branches.